# The .bin File in LLM Training

A `.bin` file is a **binary file that stores already tokenized data**: a flat, continuous array of token ids. There is no text inside, only numbers.

Because the data is pre-tokenized, the training loop does not need to run the tokenizer. It only reads blocks of ids directly from disk, which makes loading fast and reproducible.

---

## Special tokens

Usually, the `.bin` already includes the **special tokens** inserted during tokenization, most importantly the **EOS** (end-of-sequence) token placed between documents.

Since the file is a flat stream, the EOS is the **only marker** of where one document ends and the next begins.

---

## Memmap

**Memmap** (memory-mapped file) is a technique that maps a file on disk directly into the process's virtual address space. The file is accessed as if it were a regular array in memory, but the data is **not loaded all at once**. The operating system reads only the parts that are actually accessed, on demand, in small pages.

In NumPy, this is available through `np.memmap`.

### Why we use it

- **Datasets larger than RAM:** a tokenized corpus can have billions of tokens (tens or hundreds of GB). Memmap lets us index it without loading it fully.
- **Lazy loading:** only the slices we access (for example, one window of `seq_len + 1` tokens) are read from disk.
- **Fast startup:** opening the file is almost instant, since nothing is read until it is accessed.
- **Shared across processes:** multiple `DataLoader` workers can map the same file, and the OS shares the pages in its cache instead of duplicating the data in each worker.
- **Simple indexing:** we use normal array slicing (`data[idx : idx + seq_len + 1]`) with no manual file seeking or reading.

### Why it is useful for `.bin` files

A `.bin` file is a **flat array of token ids** with a fixed dtype (for example `uint16`). This is exactly the structure memmap needs:

- Every token has the same size, so the position of token `i` is simply `i × bytes_per_token`. Random access is trivial and fast.
- There is no header, compression, or parsing to deal with, so the bytes on disk map one-to-one to the array values.
- It fits the packing workflow: we only


> Typically, the number of tokens in a document can vary significantly; consequently, it is common to have leftover tokens that do not fill the `batch_size`, and when dealing with large volumes of data, we can simply discard them.

In [ ]:
import numpy as np

data = np.arange(10000) # Let's supose that our actual doc have 10k tokens
seq_len = 1024 # Let's supose that we want batchs of (batch_size, 1024), which each batch gonna have 1024 tokens

n_windows = (len(data) - 1) // seq_len # Number of windows that we can do in that document
rest = len(data) % seq_len
print(f"Number of packing that we have in this doc: {n_windows}")

for pack in range(n_windows): # Loop for each packing
    idx = seq_len * pack
    tokens = data[idx:idx + seq_len + 1].astype(np.int64) # Normally .BIN stay in uint16/32, so we need to change for int32/64 for pytorch accepts

    x, y = tokens[:seq_len], tokens[1:] # Change NUMPY for TENSOR  for the pytorch acceps # torch.to_numpy(x)


    print(f'=' * 50)
    print(f'Packing number: {pack}')
    print(f'Value of idx: {idx}')
    print(f"X: {x[:5]}, y: {y[:5]}")
    print(f'Len of x: {x.shape}, Len of y: {y.shape}')

Number of packing that we have in this doc: 9
Packing number: 0
Value of idx: 0
X: [0 1 2 3 4], y: [1 2 3 4 5]
Len of x: (1024,), Len of y: (1024,)
Packing number: 1
Value of idx: 1024
X: [1024 1025 1026 1027 1028], y: [1025 1026 1027 1028 1029]
Len of x: (1024,), Len of y: (1024,)
Packing number: 2
Value of idx: 2048
X: [2048 2049 2050 2051 2052], y: [2049 2050 2051 2052 2053]
Len of x: (1024,), Len of y: (1024,)
Packing number: 3
Value of idx: 3072
X: [3072 3073 3074 3075 3076], y: [3073 3074 3075 3076 3077]
Len of x: (1024,), Len of y: (1024,)
Packing number: 4
Value of idx: 4096
X: [4096 4097 4098 4099 4100], y: [4097 4098 4099 4100 4101]
Len of x: (1024,), Len of y: (1024,)
Packing number: 5
Value of idx: 5120
X: [5120 5121 5122 5123 5124], y: [5121 5122 5123 5124 5125]
Len of x: (1024,), Len of y: (1024,)
Packing number: 6
Value of idx: 6144
X: [6144 6145 6146 6147 6148], y: [6145 6146 6147 6148 6149]
Len of x: (1024,), Len of y: (1024,)
Packing number: 7
Value of idx: 7168
X: [7

In [13]:
resto = (len(data)) % seq_len
print(resto)

784
